
# 📘 Day 5 学习笔记：PagedAttention 与 Continuous Batching

> 今日主线：**KV Cache 很大 → 传统连续内存分配浪费严重 → PagedAttention 用 block/page 管理 KV Cache → 静态 batching 被长请求拖住 → Continuous Batching 在每个 decode step 动态补位 → Serving 吞吐提升**

---

## 🎯 今日学习大纲

### 1. 今天你要攻克的核心问题

昨天你已经知道：KV Cache 显存近似为：

\[
\text{KV Cache Bytes}=B \times 2 \times L \times T \times H_{kv} \times d_h \times \text{bytes}
\]

但在真实 serving 中，问题不只是“KV Cache 大”，还包括：

1. **请求长度不一样**：有人生成 20 tokens，有人生成 500 tokens。
2. **KV Cache 动态增长**：每生成一个 token，cache 就要追加一段。
3. **传统连续内存分配浪费严重**：如果按最大长度预分配，短请求会浪费大量显存。
4. **静态 batching 被长请求拖住**：同一批请求必须等最长请求结束，GPU 空转严重。
5. **多用户请求不断到来**：工程上必须处理队列、并发、限流、抢占与调度。

所以 Day 5 的核心不是新的 Transformer 数学公式，而是：

> **LLM Serving 的内存管理与调度问题。**

---

## ✅ 今日完成目标

学完这份 Notebook，你应该能做到：

1. 说清楚 **PagedAttention 不是改 attention 公式，而是改 KV Cache 内存管理方式**。
2. 画出 **logical blocks / physical blocks / block table** 的关系。
3. 用代码模拟 **contiguous pre-allocation** 和 **paged block allocation** 的显存浪费差异。
4. 说清楚 **Continuous Batching** 和传统 static batching 的区别。
5. 用代码模拟多请求场景下，static batching 为什么被长请求拖住。
6. 能回答面试官连续追问：
   - PagedAttention 和普通 Attention 的区别是什么？
   - 为什么 PagedAttention 能提高并发容量？
   - Continuous Batching 为什么提升吞吐？
   - vLLM 为什么比朴素 Transformers serving 快？
7. 能把今天内容映射到：
   - 多用户 RAG / Agent 请求队列
   - 并发控制
   - 限流
   - GPU token budget
   - serving 端 throughput / latency trade-off

---

## 🧠 今日关键词

- KV Cache fragmentation
- pre-allocation
- logical block
- physical block
- block table
- PagedAttention
- static batching
- continuous batching
- iteration-level scheduling
- throughput
- TTFT / TPOT
- max_num_seqs
- max_num_batched_tokens

---

## 🧭 今日学习节奏

建议顺序：

```text
1. 复习 KV Cache 为什么大
2. 理解传统连续内存分配为什么浪费
3. 学 PagedAttention 的 block table
4. 跑显存浪费模拟图
5. 理解 static batching 的长尾问题
6. 跑 continuous batching 调度模拟图
7. 最后整理面试问答和工程排障清单
```


In [ ]:

# Day 5 基础库
import math
import heapq
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (8, 4.8)
plt.rcParams['axes.unicode_minus'] = False

print('环境就绪 ✅')



---

# 1️⃣ 先复习：KV Cache 为什么是 Serving 的核心瓶颈？

昨天我们已经推过公式：

\[
\text{KV Cache Bytes}=B \times 2 \times L \times T \times H_{kv} \times d_h \times \text{bytes}
\]

其中：

- \(B\)：batch size
- \(2\)：K 和 V 两份
- \(L\)：Transformer 层数
- \(T\)：上下文长度
- \(H_{kv}\)：KV heads 数
- \(d_h\)：head dimension
- `bytes`：每个元素字节数，例如 fp16/bf16 是 2 bytes

---

## 1.1 单 token 的 KV Cache 成本

单个 token、单层的 KV cache 大小为：

\[
\text{KV bytes per token per layer}=2 \times H_{kv} \times d_h \times \text{bytes}
\]

所有层合起来：

\[
\text{KV bytes per token}=2 \times L \times H_{kv} \times d_h \times \text{bytes}
\]

这很重要，因为 PagedAttention 的 block，本质上就是把多个 token 的 KV cache 打包成固定大小的 block。

---

## 1.2 为什么 Serving 比单次本地推理更麻烦？

本地单次推理时，你通常只有一个请求：

```text
一个 prompt → 生成一段答案 → 结束
```

但是服务端是：

```text
请求 A 到了，还没结束
请求 B 到了，也要加入
请求 C 很短，很快结束
请求 D 很长，占着 cache 很久
请求 E 需要排队
```

所以服务端真正的问题是：

> **在有限 GPU 显存里，同时容纳尽可能多的请求，并让 GPU 每一步都尽量忙。**

这就是 Day 5 要解决的两个关键词：

```text
PagedAttention：解决 KV Cache 内存浪费 / 碎片问题
Continuous Batching：解决静态 batch 被长请求拖住的问题
```


In [ ]:

def kv_bytes_per_token(num_layers=32, num_kv_heads=8, head_dim=128, dtype_bytes=2):
    return 2 * num_layers * num_kv_heads * head_dim * dtype_bytes


def readable_bytes(n):
    units = ['B', 'KB', 'MB', 'GB', 'TB']
    x = float(n)
    for u in units:
        if x < 1024 or u == units[-1]:
            return f'{x:.2f} {u}'
        x /= 1024

for h_kv in [32, 8, 1]:
    b = kv_bytes_per_token(num_layers=32, num_kv_heads=h_kv, head_dim=128, dtype_bytes=2)
    print(f'H_kv={h_kv:2d}: 每个 token 的全层 KV Cache ≈ {readable_bytes(b)}')



---

# 2️⃣ 传统连续内存分配为什么浪费？

## 2.1 朴素做法：按最大长度预分配

假设服务端为了简单，给每个请求都按 `max_seq_len` 预留 KV Cache。

例如：

```text
max_seq_len = 2048 tokens
```

但是实际请求可能是：

```text
请求 A：实际用了 200 tokens
请求 B：实际用了 1800 tokens
请求 C：实际用了 80 tokens
请求 D：实际用了 600 tokens
```

如果每个请求都分配 2048 tokens 的 cache，那么短请求会浪费大量显存。

---

## 2.2 为什么不能只按当前长度精确分配？

因为 decode 阶段 KV Cache 是动态增长的：

```text
第 1 个新 token：cache 长度 +1
第 2 个新 token：cache 长度 +1
...
```

如果传统方式要求连续内存，那么动态扩容会导致：

- 需要搬迁内存
- 产生碎片
- 很难高效共享 prefix
- 很难容纳大量长短不一的请求

这和操作系统里“每个进程必须占一整块连续大内存”很像：

```text
连续分配：简单，但容易碎片化和浪费
分页分配：逻辑连续，物理可以不连续
```

PagedAttention 就是把这个思想搬到了 KV Cache 管理里。


In [ ]:

# 模拟一批请求的实际长度差异
np.random.seed(7)
num_requests = 24
max_seq_len = 2048
# 模拟真实场景：很多短请求，少量长请求
actual_lens = np.clip(np.random.lognormal(mean=6.2, sigma=0.9, size=num_requests).astype(int), 32, max_seq_len)
actual_lens.sort()
actual_lens = actual_lens[::-1]

request_df = pd.DataFrame({
    'request_id': [f'req_{i:02d}' for i in range(num_requests)],
    'actual_tokens': actual_lens,
    'preallocated_tokens': max_seq_len,
    'wasted_tokens': max_seq_len - actual_lens,
})
request_df.head(10)


In [ ]:

# 可视化：按最大长度预分配会浪费多少
x = np.arange(num_requests)

plt.figure(figsize=(10, 4.8))
plt.bar(x, request_df['preallocated_tokens'], alpha=0.35, label='预分配 tokens')
plt.bar(x, request_df['actual_tokens'], label='实际使用 tokens')
plt.xlabel('Request index')
plt.ylabel('Tokens')
plt.title('朴素连续预分配：短请求浪费大量 KV Cache 空间')
plt.legend()
plt.show()

used = request_df['actual_tokens'].sum()
allocated = request_df['preallocated_tokens'].sum()
waste = allocated - used
print(f'实际使用 tokens: {used}')
print(f'预分配 tokens: {allocated}')
print(f'浪费 tokens: {waste}')
print(f'浪费比例: {waste / allocated:.1%}')



---

# 3️⃣ PagedAttention：核心思想不是改公式，而是改 KV Cache 存储

## 3.1 普通 Attention 数学公式不变

Scaled Dot-Product Attention 仍然是：

\[
\text{Attention}(Q,K,V)=\text{softmax}\left(\frac{QK^\top}{\sqrt{d_h}}\right)V
\]

PagedAttention **没有修改这个数学公式**。

它修改的是：

> **K/V 在 GPU 显存中的组织方式。**

---

## 3.2 OS 分页类比

操作系统里：

```text
进程看到的是连续的虚拟地址空间
真实物理内存可以是不连续的页框
页表负责从虚拟页映射到物理页
```

PagedAttention 里：

```text
每个 sequence 看到的是连续的 logical KV blocks
真实 GPU 显存里是分散的 physical KV blocks
block table 负责从 logical block 映射到 physical block
```

对比：

| 操作系统 | PagedAttention |
|---|---|
| 虚拟页 virtual page | logical KV block |
| 物理页 physical page frame | physical KV block |
| 页表 page table | block table |
| 进程地址空间 | sequence 的 KV cache |

---

## 3.3 一个 sequence 的 KV Cache 如何分页？

假设 block size = 16 tokens。

一个请求当前长度为 45 tokens：

```text
logical block 0: tokens 0-15
logical block 1: tokens 16-31
logical block 2: tokens 32-44，最后一个 block 只用 13 个 token
```

需要的 block 数：

\[
\text{num\_blocks}=\left\lceil \frac{T}{\text{block\_size}} \right\rceil
\]

只会在最后一个 block 有少量内部浪费。

---

## 3.4 block 的显存大小

如果一个 block 存 \(B_s\) 个 token，那么全层 KV block 大小是：

\[
\text{Block Bytes}=B_s \times 2 \times L \times H_{kv} \times d_h \times \text{bytes}
\]

其中 \(B_s\) 是 block size。

这就是 vLLM 这类 serving 系统可以按 block 管理 cache 的基础。


In [ ]:

def blocks_needed(seq_len, block_size):
    return math.ceil(seq_len / block_size)


def paged_allocated_tokens(seq_len, block_size):
    return blocks_needed(seq_len, block_size) * block_size

block_size = 16
request_df['paged_allocated_tokens'] = request_df['actual_tokens'].apply(lambda t: paged_allocated_tokens(t, block_size))
request_df['paged_wasted_tokens'] = request_df['paged_allocated_tokens'] - request_df['actual_tokens']
request_df[['request_id', 'actual_tokens', 'preallocated_tokens', 'wasted_tokens', 'paged_allocated_tokens', 'paged_wasted_tokens']].head(10)


In [ ]:

# 对比连续预分配与 Paged block 分配的浪费
naive_alloc = request_df['preallocated_tokens'].sum()
paged_alloc = request_df['paged_allocated_tokens'].sum()
actual_used = request_df['actual_tokens'].sum()

compare = pd.DataFrame({
    '方案': ['实际使用', '朴素 max_seq_len 预分配', f'Paged block 分配 block={block_size}'],
    'tokens': [actual_used, naive_alloc, paged_alloc]
})
compare['waste_vs_actual'] = compare['tokens'] - actual_used
compare


In [ ]:

plt.figure(figsize=(8, 4.5))
plt.bar(compare['方案'], compare['tokens'])
plt.ylabel('Allocated token slots')
plt.title('Paged block allocation 大幅减少 KV Cache 预分配浪费')
plt.xticks(rotation=15, ha='right')
for i, v in enumerate(compare['tokens']):
    plt.text(i, v + max(compare['tokens']) * 0.02, str(v), ha='center')
plt.show()

print(f'朴素预分配浪费比例: {(naive_alloc - actual_used) / naive_alloc:.1%}')
print(f'Paged 分配浪费比例: {(paged_alloc - actual_used) / paged_alloc:.1%}')



---

# 4️⃣ Block Table 可视化：逻辑连续，物理不连续

## 4.1 关键理解

PagedAttention 中，一个请求的 KV Cache 在逻辑上仍然是连续的：

```text
logical block 0 → logical block 1 → logical block 2 → ...
```

但物理显存里不要求连续：

```text
logical block 0 → physical block 7
logical block 1 → physical block 2
logical block 2 → physical block 13
```

这就避免了必须找一整块连续大显存的问题。

---

## 4.2 为什么这有利于 serving？

因为多用户请求的长度是动态的：

- 有人很快结束，释放几个 block
- 有人持续生成，需要追加 block
- 新请求进来，可以复用释放出来的 block

所以 block manager 可以像内存管理器一样：

```text
分配 block
释放 block
复用 block
维护 block table
```

这就是 PagedAttention 的工程价值。


In [ ]:

# 构造一个 toy block table
random.seed(3)
seq_lengths = [45, 20, 61, 9]
block_size = 16
free_physical_blocks = list(range(20))
random.shuffle(free_physical_blocks)

block_tables = {}
for seq_id, length in enumerate(seq_lengths):
    n_blocks = blocks_needed(length, block_size)
    phys = [free_physical_blocks.pop() for _ in range(n_blocks)]
    block_tables[f'seq_{seq_id}'] = phys

block_table_rows = []
for seq, phys_blocks in block_tables.items():
    for logical_idx, physical_idx in enumerate(phys_blocks):
        block_table_rows.append({
            'sequence': seq,
            'logical_block': logical_idx,
            'physical_block': physical_idx
        })

block_table_df = pd.DataFrame(block_table_rows)
block_table_df


In [ ]:

# 可视化 block table：每个格子写 physical block id
max_blocks = max(len(v) for v in block_tables.values())
mat = np.full((len(block_tables), max_blocks), np.nan)
for i, (seq, phys) in enumerate(block_tables.items()):
    mat[i, :len(phys)] = phys

plt.figure(figsize=(8, 3.5))
plt.imshow(~np.isnan(mat), aspect='auto')
plt.yticks(range(len(block_tables)), list(block_tables.keys()))
plt.xticks(range(max_blocks), [f'logical {i}' for i in range(max_blocks)])
plt.title('Block Table：logical block → physical block')

for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        if not np.isnan(mat[i, j]):
            plt.text(j, i, f'P{int(mat[i,j])}', ha='center', va='center')
        else:
            plt.text(j, i, '-', ha='center', va='center')
plt.show()



---

# 5️⃣ PagedAttention 的面试级总结

## 5.1 一句话解释

> PagedAttention 是一种 KV Cache 内存管理方法，它把每个 sequence 的 KV Cache 拆成固定大小 block，通过 block table 把逻辑连续的 KV Cache 映射到物理上不连续的 GPU memory blocks，从而减少预分配浪费和内存碎片。

---

## 5.2 它不是在解决什么？

PagedAttention **不是**：

- 新的位置编码
- 新的 attention score 公式
- 新的模型结构
- 改变 QKᵀV 的数学计算

它解决的是：

```text
服务端 KV Cache 如何高效放进 GPU 显存
```

---

## 5.3 它为什么能提高吞吐？

因为显存利用率提高后，服务端能同时容纳更多 active sequences。

在 LLM Serving 中：

```text
能同时放进显存的请求越多
→ batch 越容易做大
→ GPU 越不容易空转
→ 吞吐越高
```

但注意：

> PagedAttention 主要解决 memory management；它和 Continuous Batching 配合，才更完整地解决 serving 吞吐问题。



---

# 6️⃣ Static Batching：为什么会被长请求拖住？

## 6.1 传统静态 batching

静态 batching 的模式是：

```text
凑一批请求
一起 decode
必须等这一批全部完成
下一批才能进来
```

类比餐厅：

```text
一桌人必须全部吃完，下一桌才能坐
```

问题是请求长度不同：

```text
请求 A：20 tokens
请求 B：500 tokens
请求 C：30 tokens
请求 D：40 tokens
```

如果它们在同一个 batch 里，那么 A/C/D 很早结束，但 GPU 的 batch 槽位会被 B 拖住。

---

## 6.2 长尾问题

静态 batch 的总耗时由最长请求决定：

\[
\text{batch decode steps}=\max_i(\text{generated tokens}_i)
\]

如果 batch 内长度差异很大，短请求对应的槽位会空出来，但新请求不能马上进来。

这就造成 GPU 利用率低。


In [ ]:

# 构造一组请求：arrival_time 和需要生成的 tokens
requests = pd.DataFrame({
    'request_id': [f'R{i}' for i in range(10)],
    'arrival_time': [0, 0, 0, 0, 2, 3, 4, 6, 6, 8],
    'gen_tokens':   [10, 60, 12, 18, 20, 8, 45, 14, 9, 30],
})
requests


In [ ]:

# Static batching 模拟：凑够 batch_size 或没有更多请求，整批一起跑到最长请求结束

def simulate_static_batching(req_df, batch_size=4):
    reqs = req_df.sort_values('arrival_time').to_dict('records')
    t = 0
    idx = 0
    finished = []
    batches = []
    waiting = []

    while idx < len(reqs) or waiting:
        # 把已到达请求放入等待队列
        while idx < len(reqs) and reqs[idx]['arrival_time'] <= t:
            waiting.append(reqs[idx])
            idx += 1

        # 如果等待队列为空，时间跳到下一个请求到达
        if not waiting and idx < len(reqs):
            t = reqs[idx]['arrival_time']
            continue

        batch = waiting[:batch_size]
        waiting = waiting[batch_size:]
        batch_start = t
        batch_duration = max(r['gen_tokens'] for r in batch)
        batch_end = batch_start + batch_duration

        for r in batch:
            finished.append({
                'request_id': r['request_id'],
                'arrival_time': r['arrival_time'],
                'gen_tokens': r['gen_tokens'],
                'start_time': batch_start,
                'finish_time': batch_end,
                'latency': batch_end - r['arrival_time'],
                'mode': 'static'
            })
        batches.append((batch_start, batch_end, [r['request_id'] for r in batch]))
        t = batch_end

    return pd.DataFrame(finished), batches

static_result, static_batches = simulate_static_batching(requests, batch_size=4)
static_result


In [ ]:

# 可视化 static batching 的请求生命周期
plt.figure(figsize=(10, 4.8))
for i, row in static_result.sort_values('request_id').reset_index(drop=True).iterrows():
    plt.barh(row['request_id'], row['finish_time'] - row['start_time'], left=row['start_time'])
    plt.text(row['finish_time'] + 0.5, i, f"lat={row['latency']}", va='center', fontsize=8)
plt.xlabel('Time steps')
plt.title('Static Batching：一批请求被最长请求拖住')
plt.grid(axis='x', alpha=0.3)
plt.show()

print('Static 平均延迟:', static_result['latency'].mean())
print('Static 总完成时间:', static_result['finish_time'].max())



---

# 7️⃣ Continuous Batching：每个 decode step 都能重新调度

## 7.1 核心思想

Continuous Batching，又叫 iteration-level batching。

它不是“一批请求一起开始，一起结束”，而是：

```text
每一轮 decode 后检查：
    谁结束了？移出去
    谁新到了？补进来
    下一轮继续跑 active batch
```

类比餐厅：

```text
谁吃完谁离席，新客人立刻补位
```

---

## 7.2 为什么适合 LLM decode？

因为 LLM decode 本来就是逐 token 的：

```text
decode step 1
decode step 2
decode step 3
...
```

每一步之间本来就有调度机会。

所以 Continuous Batching 很自然：

```text
每个 decode step 都重新组织 active requests
```

---

## 7.3 关键收益

它提高的是：

```text
GPU batch 槽位利用率
```

尤其在请求长度差异很大时，continuous batching 能明显减少空转。


In [ ]:

# Continuous batching 模拟：每个 time step 生成 1 token，空槽位可以补新请求

def simulate_continuous_batching(req_df, max_active=4):
    reqs = req_df.sort_values('arrival_time').to_dict('records')
    t = 0
    idx = 0
    waiting = []
    active = []
    finished = []
    timeline = []

    # active item: dict with remaining, start_time
    while idx < len(reqs) or waiting or active:
        # 加入所有已到达请求
        while idx < len(reqs) and reqs[idx]['arrival_time'] <= t:
            r = dict(reqs[idx])
            r['remaining'] = r['gen_tokens']
            r['start_time'] = None
            waiting.append(r)
            idx += 1

        # 补满 active slots
        while len(active) < max_active and waiting:
            r = waiting.pop(0)
            if r['start_time'] is None:
                r['start_time'] = t
            active.append(r)

        if not active:
            # 没有 active，请跳到下个请求到达时间
            if idx < len(reqs):
                t = reqs[idx]['arrival_time']
            continue

        timeline.append({'time': t, 'active': [r['request_id'] for r in active]})

        # 每个 active request 生成一个 token
        for r in active:
            r['remaining'] -= 1

        t += 1

        # 移除完成请求
        still_active = []
        for r in active:
            if r['remaining'] <= 0:
                finished.append({
                    'request_id': r['request_id'],
                    'arrival_time': r['arrival_time'],
                    'gen_tokens': r['gen_tokens'],
                    'start_time': r['start_time'],
                    'finish_time': t,
                    'latency': t - r['arrival_time'],
                    'mode': 'continuous'
                })
            else:
                still_active.append(r)
        active = still_active

    return pd.DataFrame(finished), timeline

cont_result, cont_timeline = simulate_continuous_batching(requests, max_active=4)
cont_result.sort_values('request_id')


In [ ]:

# 可视化 continuous batching 的请求生命周期
plt.figure(figsize=(10, 4.8))
for i, row in cont_result.sort_values('request_id').reset_index(drop=True).iterrows():
    plt.barh(row['request_id'], row['finish_time'] - row['start_time'], left=row['start_time'])
    plt.text(row['finish_time'] + 0.5, i, f"lat={row['latency']}", va='center', fontsize=8)
plt.xlabel('Time steps')
plt.title('Continuous Batching：请求完成后立即补位')
plt.grid(axis='x', alpha=0.3)
plt.show()

print('Continuous 平均延迟:', cont_result['latency'].mean())
print('Continuous 总完成时间:', cont_result['finish_time'].max())


In [ ]:

# 对比 static 与 continuous 的延迟
merged = static_result[['request_id', 'latency']].rename(columns={'latency': 'static_latency'}).merge(
    cont_result[['request_id', 'latency']].rename(columns={'latency': 'continuous_latency'}),
    on='request_id'
)
merged['latency_reduction'] = merged['static_latency'] - merged['continuous_latency']
merged


In [ ]:

plt.figure(figsize=(9, 4.5))
x = np.arange(len(merged))
width = 0.38
plt.bar(x - width/2, merged['static_latency'], width, label='Static')
plt.bar(x + width/2, merged['continuous_latency'], width, label='Continuous')
plt.xticks(x, merged['request_id'])
plt.ylabel('Latency')
plt.title('Static vs Continuous：同一批请求的延迟对比')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

summary = pd.DataFrame({
    'mode': ['static', 'continuous'],
    'avg_latency': [static_result['latency'].mean(), cont_result['latency'].mean()],
    'makespan': [static_result['finish_time'].max(), cont_result['finish_time'].max()],
})
summary


In [ ]:

# 可视化 continuous batching 每个 step 的 active batch size
active_sizes = [len(item['active']) for item in cont_timeline]
times = [item['time'] for item in cont_timeline]

plt.figure(figsize=(9, 3.8))
plt.step(times, active_sizes, where='post')
plt.xlabel('Time step')
plt.ylabel('Active requests')
plt.title('Continuous Batching 每步动态维持 active batch')
plt.ylim(0, 5)
plt.grid(alpha=0.3)
plt.show()



---

# 8️⃣ PagedAttention + Continuous Batching 为什么要配合？

## 8.1 单独 PagedAttention 解决什么？

解决：

```text
KV Cache 怎么更省显存、更少碎片地放进 GPU
```

它让同样显存能容纳更多 active sequences。

---

## 8.2 单独 Continuous Batching 解决什么？

解决：

```text
每个 decode step 如何动态补位，让 GPU batch 不空
```

它让 GPU 更持续地处于忙碌状态。

---

## 8.3 二者合起来

```text
PagedAttention：让更多请求放得下
Continuous Batching：让这些请求调度得更满
```

于是 serving 吞吐提升。

---

## 8.4 和 Day 4 的关系

Day 4 你学的是：

```text
为什么 KV Cache 必须存在，以及它怎么算显存
```

Day 5 你学的是：

```text
当很多用户同时来了，KV Cache 应该怎么被管理和调度
```

所以 Day 4 是“单请求 decode 原理”，Day 5 是“多请求 serving 系统”。



---

# 9️⃣ Chunked Prefill / Prefix Caching：了解即可

虽然今天核心是 PagedAttention 和 Continuous Batching，但你会经常在 vLLM、TGI、SGLang 里看到另外两个词。

## 9.1 Chunked Prefill

问题：

```text
一个超长 prompt 的 prefill 可能占用很长时间，阻塞 decode 请求
```

Chunked Prefill 的思想：

```text
把长 prompt 的 prefill 切成更小的 token chunks
穿插调度 prefill 和 decode
减少 decode 被长 prefill 饿死的问题
```

## 9.2 Prefix Caching

问题：

```text
很多请求共享相同系统提示词、工具描述、RAG 模板前缀
```

Prefix Caching 的思想：

```text
相同 prefix 的 KV Cache 不重复计算
直接复用已有 cache
```

这对于这些场景很有用：

- 多轮对话
- Agent 系统提示词很长
- RAG 模板固定
- 同一应用大量请求共享 prompt 前缀

---

## 今天不需要深挖它们

你只要记住：

```text
PagedAttention 管 cache 内存
Continuous Batching 管请求调度
Chunked Prefill 管长 prompt prefill 阻塞
Prefix Caching 管重复前缀复用
```



---

# 🔟 工程映射：多用户 RAG / Agent 请求怎么设计？

你的路线要求 Day 5 映射到：

```text
多用户 RAG / Agent 请求的排队、并发、限流直觉
```

下面是工程视角。

---

## 10.1 RAG / Agent 请求为什么比普通 Chat 更难调度？

普通 Chat：

```text
用户问题 → LLM 生成
```

RAG：

```text
用户问题 → query rewrite → retrieval → rerank → prompt 拼接 → LLM 生成
```

Agent：

```text
用户问题 → LLM 思考 → tool call → 观察结果 → 再思考 → 再 tool call → 最终回答
```

所以 Agent / RAG 的请求长度、耗时和 token 数更不稳定。

这会导致：

- 有些请求 prompt 很长
- 有些请求 tool call 慢
- 有些请求生成短
- 有些请求生成长
- 有些请求需要多轮模型调用

这正是 Continuous Batching 发挥作用的场景。

---

## 10.2 Serving 层你需要关注的参数

常见参数包括：

```text
max_model_len
max_num_seqs
max_num_batched_tokens
gpu_memory_utilization
kv_cache_dtype
enable_prefix_caching
chunked_prefill
```

这些参数背后的直觉：

| 参数 | 影响 |
|---|---|
| max_model_len | 单请求最大上下文，直接影响 KV Cache 上限 |
| max_num_seqs | 同时活跃 sequence 数 |
| max_num_batched_tokens | 一个 step 内允许处理的 token 总数 |
| gpu_memory_utilization | 分给 cache / runtime 的显存比例 |
| kv_cache_dtype | KV Cache 每个元素字节数 |
| enable_prefix_caching | 重复前缀复用 |
| chunked_prefill | 防止长 prompt prefill 阻塞 decode |

---

## 10.3 面试时怎么把它讲成系统设计？

你可以这样讲：

> 在多用户 RAG / Agent serving 中，瓶颈往往不是单个矩阵乘法，而是请求长度分布不均、KV Cache 显存占用大、静态 batch 被长请求拖住。PagedAttention 用分页式 KV Cache 管理减少碎片和浪费，Continuous Batching 在每个 decode iteration 动态补位，提高 GPU batch 利用率。工程上还要配合 max_num_seqs、max_num_batched_tokens、限流、队列优先级和超时策略，避免少量长请求拖垮整体服务。



---

# 1️⃣1️⃣ 原理级排障场景

## 场景 1：vLLM 服务 QPS 上不去，但 GPU 利用率也不高

可能原因：

```text
1. max_num_seqs 太小，active batch 不够大
2. max_num_batched_tokens 太小，每步 token budget 不够
3. 请求长度差异大，调度效率差
4. 长 prefill 阻塞 decode
5. CPU tokenizer / 后处理成为瓶颈
6. 网络 streaming 太慢，反压到后端
7. KV Cache block 不足，出现等待或抢占
```

排查顺序：

```text
1. 看 request length 分布
2. 看 prefill latency / decode latency
3. 看 active batch size 曲线
4. 看 KV cache usage
5. 看 GPU utilization
6. 看 CPU tokenizer 和网络发送耗时
```

---

## 场景 2：并发一高就 OOM

可能原因：

```text
1. max_model_len 设置过高
2. max_num_seqs 设置过高
3. 模型 H_kv 较大，KV Cache 重
4. kv_cache_dtype 没压缩
5. prefix caching / block reuse 没启用或没命中
6. 请求 prompt 太长，RAG 塞入上下文过多
```

解决方向：

```text
1. 降 max_model_len
2. 降 max_num_seqs
3. 控制 RAG chunk 数量和 prompt 长度
4. 使用 GQA/MQA 模型
5. 使用更低精度 KV Cache，前提是框架支持
6. 打开 prefix caching / chunked prefill
```

---

## 场景 3：短请求被长请求拖慢

可能原因：

```text
1. 静态 batch 或调度策略不合理
2. 长 prompt prefill 阻塞 decode
3. 没有请求优先级或队列隔离
4. 长短请求混在同一个服务池
```

解决方向：

```text
1. 使用 continuous batching
2. 开 chunked prefill
3. 短请求和长请求分池
4. 设置 max_new_tokens 上限
5. 给交互式请求更高优先级
```



---

# 1️⃣2️⃣ 面试高频问答

## Q1：PagedAttention 和普通 Attention 的区别是什么？

**满分回答：**

PagedAttention 不改变 attention 的数学公式，仍然是 \(\text{softmax}(QK^\top/\sqrt{d})V\)。它改变的是 KV Cache 的内存管理方式：把每个 sequence 的 KV Cache 切成固定大小 block，通过 block table 映射到物理上不连续的 GPU memory blocks，从而减少连续预分配带来的浪费和碎片。

---

## Q2：PagedAttention 为什么能提高吞吐？

**满分回答：**

它提高吞吐不是因为单次 attention 算得更少，而是因为 KV Cache 显存利用率更高，能同时容纳更多 active sequences。可并发请求变多后，batch 更容易做大，GPU 更不容易空转，所以整体 serving throughput 提升。

---

## Q3：Continuous Batching 为什么适合 LLM？

**满分回答：**

LLM decode 天然是逐 token 的，每个 decode step 后都有重新调度的机会。Continuous Batching 在每一步把已完成请求移出，把新请求补入 active batch，避免静态 batch 被最长请求拖住，从而提升 GPU batch 利用率和吞吐。

---

## Q4：PagedAttention 和 Continuous Batching 分别解决什么问题？

**满分回答：**

PagedAttention 解决 KV Cache 内存管理问题，减少碎片和预分配浪费；Continuous Batching 解决请求调度问题，减少长短请求混合时的 GPU 空转。前者让更多请求放得下，后者让这些请求跑得更满。

---

## Q5：为什么长请求会拖慢短请求？

**满分回答：**

在静态 batching 中，一个 batch 的生命周期由最长请求决定。短请求虽然早完成，但新请求无法立刻补位，导致 batch 槽位空转。Continuous Batching 通过 iteration-level scheduling 缓解这个长尾问题。

---

## Q6：vLLM 为什么比朴素 Transformers serving 快？

**满分回答：**

核心原因是 vLLM 面向 serving 做了系统级优化，包括 PagedAttention 管理 KV Cache、Continuous Batching 动态调度请求、以及 prefix caching、chunked prefill、优化 kernel 等。它不是单纯模型更小或 attention 公式不同，而是把 GPU 显存和调度资源利用得更充分。

---

## Q7：如果服务 QPS 上不去，你会怎么排查？

**满分回答：**

先区分是 prefill 瓶颈还是 decode 瓶颈；再看 active batch size、KV cache usage、GPU utilization、request length 分布、max_num_seqs、max_num_batched_tokens、CPU tokenizer、后处理和网络 streaming。LLM serving 的瓶颈经常是调度和内存，而不是单纯算力。



---

# 1️⃣3️⃣ 今日最小作业

你的 7 天路线里 Day 5 的最低代码实验是：

> **模拟 static batching vs continuous batching**

这份 Notebook 已经给了一个基础版本。你今天的作业是改参数观察变化。

---

## 作业 1：修改请求长度分布

把请求改成：

```text
很多短请求 + 1 个极长请求
```

观察 static batching 和 continuous batching 差距会不会变大。

---

## 作业 2：修改 max_active / batch_size

试试：

```text
batch_size = 2 / 4 / 8
max_active = 2 / 4 / 8
```

观察平均延迟和总完成时间变化。

---

## 作业 3：修改 block_size

试试 PagedAttention 模拟里的：

```text
block_size = 8 / 16 / 32 / 64
```

观察 block 越大，内部浪费如何变化。

---

## 作业 4：用一句话总结

请你不看笔记，口述这两句话：

```text
PagedAttention 解决什么问题？
Continuous Batching 解决什么问题？
```

---

## 作业 5：工程映射

写 10 行以内回答：

```text
如果我要部署一个多用户 RAG 服务，为什么不能只用普通 for-loop 调 model.generate？
```



---

# 1️⃣4️⃣ 推荐学习资源 / 视频 / 仓库（3-5 个高质量）

## 资源 1：vLLM 官方文档
- 类型：官方文档
- 重点看：PagedAttention、Continuous Batching、Prefix Caching、Chunked Prefill
- 链接：https://docs.vllm.ai/

## 资源 2：PagedAttention 原论文
- 标题：Efficient Memory Management for Large Language Model Serving with PagedAttention
- 类型：论文 / arXiv
- 重点看：KV Cache memory waste、block table、throughput improvement
- 链接：https://arxiv.org/abs/2309.06180

## 资源 3：SOSP 2023 PagedAttention 论文报告视频
- 类型：YouTube / 会议报告
- 适合用途：快速理解论文动机和系统设计
- 链接：https://www.youtube.com/watch?v=UdNocRPQS3Y

## 资源 4：LLM Inference Engines: vLLM, KV Cache, Paged Attention
- 类型：YouTube 视频
- 适合用途：从工程视角理解 vLLM / KV Cache / PagedAttention
- 链接：https://www.youtube.com/watch?v=DNrIu_EZz5k

## 资源 5：Anyscale Continuous Batching 文章
- 类型：工程博客
- 适合用途：理解 continuous batching 为什么提升吞吐，以及传统 batching 的低效点
- 链接：https://www.anyscale.com/blog/continuous-batching-llm-inference

---

## ✅ 今日一句话总总结

> **PagedAttention 通过分页式 block 管理减少 KV Cache 内存浪费，Continuous Batching 通过每个 decode step 动态补位减少 GPU 空转；前者让更多请求放得下，后者让 GPU 跑得更满，二者共同构成现代高吞吐 LLM Serving 的核心直觉。**
